# Waymo scenario timestep-0 BEV export (ego-centered)

이 노트북은 TFRecord에서 scenario를 하나씩 로드하고, 각 scenario의 `timestep=0` 상태를 **ego 기준 좌표계**(ego를 `(0,0)`에 고정)로 BEV 이미지로 저장합니다.

- scenario load 방식은 `simulation_utils/goal_reaching.py`의 패턴(`DatasetConfig` + `_load_scenario_state_batch_fast`)을 따릅니다.
- 렌더링은 이 노트북 안에서 새로 정의한 함수 `render_bev_t0_ego_centered`를 사용합니다.

In [1]:
from __future__ import annotations

import dataclasses
import math
import sys
from pathlib import Path
from glob import glob

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import numpy as np
from tqdm import tqdm

from waymax import config as waymax_config

# Ensure repo root imports work in notebook execution context.
REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from viz.render import _load_scenario_state_batch_fast  # noqa: E402
from viz import viz as viz_module  # noqa: E402

In [2]:
def _get_ego_index_and_anchor_t(state_batch, batch_idx: int = 0) -> tuple[int, int]:
    """Returns ego object index and reference timestep for centering."""
    is_sdc = np.asarray(state_batch.object_metadata.is_sdc[batch_idx]).astype(bool)
    ego_candidates = np.flatnonzero(is_sdc)
    if ego_candidates.size != 1:
        raise ValueError(f"Expected exactly one ego(SDC), got {ego_candidates.size}.")
    ego_idx = int(ego_candidates[0])

    ego_valid = np.asarray(state_batch.log_trajectory.valid[batch_idx, ego_idx]).astype(bool)
    if ego_valid.size == 0:
        return ego_idx, 0
    if ego_valid[0]:
        return ego_idx, 0
    valid_steps = np.flatnonzero(ego_valid)
    anchor_t = int(valid_steps[0]) if valid_steps.size > 0 else 0
    return ego_idx, anchor_t


def _transform_world_to_ego(xy_t2: np.ndarray, ego_xy_2: np.ndarray, ego_yaw: float, align_heading: bool) -> np.ndarray:
    """Converts world xy -> ego-centered xy.

    If align_heading is True, ego heading is aligned to +y axis.
    """
    rel = np.asarray(xy_t2, dtype=np.float32) - np.asarray(ego_xy_2, dtype=np.float32)[None, :]
    if not align_heading:
        return rel
    rot_rad = (0.5 * math.pi) - float(ego_yaw)
    c = math.cos(rot_rad)
    s = math.sin(rot_rad)
    rot = np.array([[c, -s], [s, c]], dtype=np.float32)
    return rel @ rot.T


def _transform_xy_any_shape(xy: np.ndarray, ego_xy_2: np.ndarray, ego_yaw: float, align_heading: bool) -> np.ndarray:
    """Transforms [...,2] world coordinates to ego-centered coordinates."""
    xy_arr = np.asarray(xy, dtype=np.float32)
    flat = xy_arr.reshape(-1, 2)
    flat_local = _transform_world_to_ego(flat, ego_xy_2, ego_yaw, align_heading)
    return flat_local.reshape(xy_arr.shape)


def _replace_xy_fields_safe(obj, xy_local: np.ndarray):
    """Replaces x/y fields if they exist; avoids passing unsupported kwargs like 'xy'."""
    kwargs = {}
    if hasattr(obj, "x"):
        kwargs["x"] = np.asarray(xy_local)[..., 0]
    if hasattr(obj, "y"):
        kwargs["y"] = np.asarray(xy_local)[..., 1]
    if kwargs:
        return obj.replace(**kwargs)
    return obj


def _plot_lane_centerlines_by_tl_control(
    ax,
    roadgraph_local,
    traffic_lights_local,
    *,
    timestep: int,
    red_states: tuple[int, ...] = (4,),
) -> None:
    """Plots all lane centerlines with TL-aware coloring.

    - Lanes controlled by red traffic lights: light red
    - All other lane centerlines: light green
    """
    rg_valid = np.asarray(roadgraph_local.valid).astype(bool)
    rg_ids = np.asarray(roadgraph_local.ids)
    rg_types = np.asarray(roadgraph_local.types)
    rg_xy = np.asarray(roadgraph_local.xy)

    lane_center_mask = np.isin(rg_types, np.array([1, 2, 3], dtype=rg_types.dtype))
    lane_mask = rg_valid & lane_center_mask
    if not np.any(lane_mask):
        return

    lane_ids = np.unique(rg_ids[lane_mask])

    red_controlled_lane_ids = set()
    if traffic_lights_local is not None:
        tl_valid_t = np.asarray(traffic_lights_local.valid[:, timestep]).astype(bool)
        tl_state_t = np.asarray(traffic_lights_local.state[:, timestep])
        tl_lane_id_t = np.asarray(traffic_lights_local.lane_ids[:, timestep])
        red_mask = tl_valid_t & np.isin(tl_state_t, np.asarray(red_states, dtype=tl_state_t.dtype)) & (tl_lane_id_t > 0)
        red_controlled_lane_ids = set(np.asarray(tl_lane_id_t[red_mask]).tolist())

    for lane_id in lane_ids:
        this_lane = lane_mask & (rg_ids == lane_id)
        lane_xy = rg_xy[this_lane]
        if lane_xy.shape[0] < 2:
            continue
        if int(lane_id) in red_controlled_lane_ids:
            color = "#ff9aa2"  # light red
            z = 4.2
        else:
            color = "#b7e4c7"  # light green
            z = 4.0
        ax.plot(
            lane_xy[:, 0],
            lane_xy[:, 1],
            color=color,
            linewidth=2.2,
            alpha=0.9,
            solid_capstyle="round",
            zorder=z,
        )


def _vehicle_box_polygon(center_xy: np.ndarray, yaw: float, length: float, width: float) -> np.ndarray:
    """Returns 4-corner polygon for a vehicle box in local plot coordinates."""
    hl = 0.5 * float(length)
    hw = 0.5 * float(width)
    corners = np.array([[hl, hw], [hl, -hw], [-hl, -hw], [-hl, hw]], dtype=np.float32)
    c = math.cos(float(yaw))
    s = math.sin(float(yaw))
    rot = np.array([[c, -s], [s, c]], dtype=np.float32)
    return corners @ rot.T + np.asarray(center_xy, dtype=np.float32)[None, :]


def render_bev_t0_ego_centered(
    state_batch,
    *,
    out_path: Path,
    batch_idx: int = 0,
    timestep: int = 0,
    front_x: float = 30.0,
    back_x: float = 30.0,
    left_y: float = 30.0,
    right_y: float = 30.0,
    align_heading: bool = True,
    draw_roadgraph: bool = True,
    draw_stop_controlled_lanes: bool = True,
    dpi: int = 200,
    figsize: tuple[float, float] = (8.0, 8.0),
) -> None:
    """Renders timestep scene into ego-centered BEV and saves PNG."""
    t_max = int(np.asarray(state_batch.log_trajectory.x).shape[-1]) - 1
    t = int(np.clip(timestep, 0, t_max))

    state_b = viz_module._index_pytree(state_batch, batch_idx)
    ego_idx, anchor_t = _get_ego_index_and_anchor_t(state_batch, batch_idx=batch_idx)
    ego_ref_t = t if np.asarray(state_batch.log_trajectory.valid[batch_idx, ego_idx, t]).astype(bool) else anchor_t

    ego_xy = np.asarray(state_batch.log_trajectory.xy[batch_idx, ego_idx, ego_ref_t], dtype=np.float32)
    ego_yaw = float(np.asarray(state_batch.log_trajectory.yaw[batch_idx, ego_idx, ego_ref_t]))

    obj_xy = np.asarray(state_batch.log_trajectory.xy[batch_idx, :, t, :], dtype=np.float32)
    obj_yaw = np.asarray(state_batch.log_trajectory.yaw[batch_idx, :, t], dtype=np.float32)
    obj_len = np.asarray(state_batch.log_trajectory.length[batch_idx, :, t], dtype=np.float32)
    obj_wid = np.asarray(state_batch.log_trajectory.width[batch_idx, :, t], dtype=np.float32)
    obj_valid = np.asarray(state_batch.log_trajectory.valid[batch_idx, :, t]).astype(bool)

    is_ego = np.asarray(state_batch.object_metadata.is_sdc[batch_idx]).astype(bool)
    xy_ego = _transform_world_to_ego(obj_xy, ego_xy, ego_yaw, align_heading=align_heading)

    fig, ax = plt.subplots(1, 1, figsize=figsize)

    # Build ego-centered roadgraph/tl copies and render with the same functions used by viz pipeline.
    if draw_roadgraph and hasattr(state_b, "roadgraph_points") and state_b.roadgraph_points is not None:
        rg_world_xy = np.stack(
            [np.asarray(state_b.roadgraph_points.x), np.asarray(state_b.roadgraph_points.y)],
            axis=-1,
        )
        rg_xy_local = _transform_xy_any_shape(
            rg_world_xy, ego_xy, ego_yaw, align_heading
        )
        roadgraph_local = _replace_xy_fields_safe(state_b.roadgraph_points, rg_xy_local)

        if hasattr(state_b, "log_traffic_light") and state_b.log_traffic_light is not None:
            tl_world_xy = np.stack(
                [np.asarray(state_b.log_traffic_light.x), np.asarray(state_b.log_traffic_light.y)],
                axis=-1,
            )
            tl_xy_local = _transform_xy_any_shape(
                tl_world_xy, ego_xy, ego_yaw, align_heading
            )
            traffic_lights_local = _replace_xy_fields_safe(state_b.log_traffic_light, tl_xy_local)
        else:
            traffic_lights_local = None

        # Draw full roadgraph first (edge, stop sign, crosswalk, etc.).
        n_before = len(ax.lines)
        viz_module.plot_roadgraph_points(ax, roadgraph_local, verbose=False)
        for line in ax.lines[n_before:]:
            line.set_alpha(1.0)
            line.set_linewidth(max(1.4, line.get_linewidth()))

        # Overlay all lane centerlines with TL-aware colors.
        _plot_lane_centerlines_by_tl_control(
            ax,
            roadgraph_local,
            traffic_lights_local,
            timestep=t,
            red_states=(4,),
        )

        if traffic_lights_local is not None:
            if draw_stop_controlled_lanes:
                viz_module.plot_stop_controlled_lanes(
                    ax, roadgraph_local, traffic_lights_local, timestep=t
                )
            viz_module.plot_traffic_light_signals_as_points(
                ax, traffic_lights_local, timestep=t, verbose=False
            )

    for n in range(obj_xy.shape[0]):
        if not obj_valid[n]:
            continue
        center = xy_ego[n]
        if align_heading:
            yaw_local = float(obj_yaw[n] - ego_yaw + 0.5 * math.pi)
        else:
            yaw_local = float(obj_yaw[n])
        poly = _vehicle_box_polygon(center, yaw_local, float(obj_len[n]), float(obj_wid[n]))
        face = "tab:red" if is_ego[n] else "tab:blue"
        alpha = 0.95 if is_ego[n] else 0.72
        patch = Polygon(poly, closed=True, facecolor=face, edgecolor="black", linewidth=0.55, alpha=alpha, zorder=6)
        ax.add_patch(patch)

    # Ego origin marker.
    ax.scatter([0.0], [0.0], c="yellow", edgecolors="black", s=80, zorder=8)

    ax.set_xlim(-float(back_x), float(front_x))
    ax.set_ylim(-float(right_y), float(left_y))
    ax.set_aspect("equal", adjustable="box")
    # ax.set_xlabel("x (m), ego-centered")
    # ax.set_ylabel("y (m), ego-centered")
    # ax.set_title(f"t={t} | ego@origin | align_heading={align_heading}")

    # Grid in ego-centered metric coordinates.
    ax.set_xticks(np.arange(-int(back_x // 10) * 10, int(front_x // 10) * 10 + 1, 10))
    ax.set_yticks(np.arange(-int(right_y // 10) * 10, int(left_y // 10) * 10 + 1, 10))
    ax.grid(True, which="major", alpha=0.35, linestyle="-")
    ax.minorticks_on()
    ax.grid(True, which="minor", alpha=0.15, linestyle=":")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(out_path, dpi=int(dpi))
    plt.close(fig)

In [4]:
# -------- User config --------
TFRECORD_DIR = Path("/zfsauton/datasets/womd/tf_example/training")  # directory containing TFRecord shards
OUTPUT_DIR = Path("test_results/bev_t0_ego_centered")

MAX_NUM_OBJECTS = 128
MAX_SCENARIOS = 10   # e.g., 100 to limit total exports
START_SCENARIO_INDEX = 0

FRONT_X = 30.0
BACK_X = 30.0
LEFT_Y = 30.0
RIGHT_Y = 30.0
ALIGN_HEADING = True

tfrecord_paths = sorted(glob(str(TFRECORD_DIR / "*")))
if not tfrecord_paths:
    raise FileNotFoundError(f"No TFRecord files found under: {TFRECORD_DIR}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

saved_count = 0
for tfrecord_path in tfrecord_paths:
    ds_cfg = dataclasses.replace(
        waymax_config.WOD_1_3_1_TRAINING,
        path=str(tfrecord_path),
        max_num_objects=int(MAX_NUM_OBJECTS),
        batch_dims=(1,),
        shuffle_seed=0,
    )

    scenario_idx = int(START_SCENARIO_INDEX)
    while True:
        if (MAX_SCENARIOS is not None) and (saved_count >= int(MAX_SCENARIOS)):
            break
        try:
            state_batch, _ = _load_scenario_state_batch_fast(ds_cfg, [scenario_idx])
        except Exception:
            # End of scenarios for this tfrecord.
            break

        tfrecord_stem = Path(tfrecord_path).name
        out_file = OUTPUT_DIR / f"{tfrecord_stem}.scenario_{scenario_idx:05d}.t0.png"
        render_bev_t0_ego_centered(
            state_batch,
            out_path=out_file,
            timestep=0,
            front_x=FRONT_X,
            back_x=BACK_X,
            left_y=LEFT_Y,
            right_y=RIGHT_Y,
            align_heading=ALIGN_HEADING,
            draw_roadgraph=True,
        )

        saved_count += 1
        if saved_count % 20 == 0:
            print(f"Saved {saved_count} images...")
        scenario_idx += 1

    if (MAX_SCENARIOS is not None) and (saved_count >= int(MAX_SCENARIOS)):
        break

print(f"Done. Saved {saved_count} BEV images to: {OUTPUT_DIR}")

Done. Saved 10 BEV images to: test_results/bev_t0_ego_centered
